In [2]:
# Some messy code to calculate relationships between lone-pair energies and optimal bonded-pair-only energies.

import json
import csv
import pandas as pd
import numpy as np

with open('data/result_vsepr_cli.json') as f:
    data = json.load(f)

rows = []
for obj in data:
    for run in obj['runs']:
        rows.append({
            'bonded_points': obj['bonded_points'],
            'lone_points': obj['lone_points'],
            'energy': run['energy']
        })

vsepr = pd.DataFrame(rows)
optimal = pd.read_csv('data/energies.csv', sep=',')

optimal

,n,energy
0,2,0.500000
1,3,1.732051
2,4,3.674235
3,5,6.474691
4,6,9.985281
5,7,14.452977
6,8,19.675288
7,9,25.759987
8,10,32.716949
9,11,40.596451


In [12]:
min_vsepr = vsepr[vsepr['lone_points'] == 0].groupby('bonded_points').min().reset_index()[:54]
min_vsepr['energy_diff'] = min_vsepr.apply(
    lambda row: row['energy'] - optimal.loc[optimal['n'] == row['bonded_points'], 'energy'].item(),
    axis=1
).clip(lower=0).round(8)


ValueError: can only convert an array of size 1 to a Python scalar

In [4]:
with open('data/lone_pair_energies.json') as f:
    data = json.load(f)

rows = []
for obj in data:
    rows.append({
        'bonded_points': obj['bonded_points'],
        'lone_points': obj['lone_points'],
        'energy': obj['best_energy']
    })

lone_vsepr = pd.DataFrame(rows)
lone_vsepr

,bonded_points,lone_points,energy
0,2,3,8.360773
1,2,4,13.286213
2,2,5,19.590620
3,3,3,12.402231
4,3,4,18.470925
...,...,...,...
148,0,4,5.582045
149,0,5,9.836612
150,1,3,5.051613
151,1,4,9.078569


In [5]:
lone_2 = vsepr[vsepr['lone_points'] == 2].groupby('bonded_points').min().sort_values('bonded_points').reset_index()[['bonded_points', 'energy']][:51]
lone_3 = lone_vsepr[lone_vsepr['lone_points'] == 3][['bonded_points', 'energy']].sort_values('bonded_points').reset_index()
lone_4 = lone_vsepr[lone_vsepr['lone_points'] == 4][['bonded_points', 'energy']].sort_values('bonded_points').reset_index()
lone_5 = lone_vsepr[lone_vsepr['lone_points'] == 5][['bonded_points', 'energy']].sort_values('bonded_points').reset_index()

In [18]:
diff_df = optimal[['n', 'energy']].rename(columns={'energy': 'energy_0'})[:54]
diff_df['two_diff'] = np.concat([lone_2['energy'].values - diff_df[:51]['energy_0'].values, [0] * 3]).astype('double')
diff_df['three_diff'] = np.concat([[0] * 1, lone_3['energy'].values - diff_df[1:52]['energy_0'].values, [0] * 2]).astype('double')
diff_df['four_diff'] = np.concat([[0] * 2, lone_4['energy'].values - diff_df[2:53]['energy_0'].values, [0] * 1]).astype('double')
diff_df['five_diff'] = np.concat([[0] * 3, lone_5['energy'].values - diff_df[3:54]['energy_0'].values]).astype('double')

diff_df

,n,energy_0,two_diff,three_diff,four_diff,five_diff
0,2,0.500000,0.259620,0.000000,0.000000,0.000000
1,3,1.732051,0.566070,0.899351,0.000000,0.000000
2,4,3.674235,0.882916,1.377379,1.907811,0.000000
3,5,6.474691,1.220170,1.886081,2.603877,3.361921
4,6,9.985281,1.573414,2.416949,3.300931,4.223559
5,7,14.452977,1.904171,2.940586,4.017947,5.137642
6,8,19.675288,2.301060,3.513489,4.762230,6.066017
7,9,25.759987,2.669705,4.067977,5.509204,6.989321
8,10,32.716949,3.052208,4.640157,6.271129,7.945559
9,11,40.596451,3.424161,5.202469,7.028813,8.897700


In [21]:
from scipy.stats import linregress

two_lone_stats = linregress(diff_df['n'][:51].values, diff_df['two_diff'][:51].values)
three_lone_stats = linregress(diff_df['n'][1:52].values, diff_df['three_diff'][1:52].values)
four_lone_stats = linregress(diff_df['n'][2:53].values, diff_df['four_diff'][2:53].values)
five_lone_stats = linregress(diff_df['n'][3:54].values, diff_df['five_diff'][3:54].values)


In [22]:
for i in [two_lone_stats, three_lone_stats, four_lone_stats, five_lone_stats]:
    print(f'y = {i.slope:.8f}x + {i.intercept:.8f}, r={i.rvalue:.4f}, p={i.pvalue:.4e}, with error {i.stderr:.8f}.')

y = 0.40801029x + -1.02874777, r=0.9997, p=5.8669e-81, with error 0.00140479.
y = 0.61478526x + -1.55621878, r=0.9998, p=1.1603e-83, with error 0.00186404.
y = 0.82298731x + -2.08135991, r=0.9998, p=6.4415e-86, with error 0.00224426.
y = 1.03222528x + -2.59482775, r=0.9998, p=6.8041e-88, with error 0.00256514.


In [10]:
# Regression on the slopes
slope_regression = linregress([2, 3, 4, 5], [two_lone_stats.slope, three_lone_stats.slope, four_lone_stats.slope, five_lone_stats.slope])
print(f'Slope regression: y = {slope_regression.slope:.8f}x + {slope_regression.intercept:.8f}, r={slope_regression.rvalue:.4f}, p={slope_regression.pvalue:.4e}, with error {slope_regression.stderr:.8f}.')

Slope regression: y = 0.20808470x + -0.00879443, r=1.0000, p=3.5202e-06, with error 0.00039042.


α (mn term): 0.15755964
β (n² term): 0.16774382
R²: 0.690811
